# Milestone 2 — Real Baseline Reproduction: RAG Robustness Under Spurious Grounding Features
### Course: 4th-Year Undergraduate NLP Research Project (Research Track)
**Anchor Paper Reference:** *Quantifying and Improving the Robustness of Retrieval-Augmented Language Models Against Spurious Features in Grounding Data* (arXiv:2503.05587, ACL 2026)  
**Author:** Siddhartha Shukla  
**Repository:** [https://github.com/SiddharthaShukla8/NLP_Project](https://github.com/SiddharthaShukla8/NLP_Project)  
**Target Compute:** Google Colab / Kaggle Free GPU (T4 / P100)

## Section 1: Verification of Environment & Dependencies
We install the authentic machine learning packages (`transformers`, `datasets`, `evaluate`, `accelerate`, `torch`) and verify that GPU acceleration is active.

In [8]:
%pip install -q transformers datasets evaluate accelerate torch

import os
import sys
import json
import re
import torch
import evaluate
import transformers
import datasets
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
from datasets import load_dataset

print("=== VERIFIED ENVIRONMENT RUNTIME ===")
print("PyTorch Version     :", torch.__version__)
print("Transformers Version:", transformers.__version__)
print("Datasets Version    :", datasets.__version__)
print("Evaluate Version    :", evaluate.__version__)

# Check GPU availability
if torch.cuda.is_available():
    device = "cuda"
    print("Running on GPU:", torch.cuda.get_device_name(0))
elif torch.backends.mps.is_available():
    device = "mps"
    print("Running on Apple Silicon MPS")
else:
    device = "cpu"
    print("Warning: No GPU detected. Running on CPU — inference will be slow.")
print(f"Active Device: {device}")

Note: you may need to restart the kernel to use updated packages.
=== VERIFIED ENVIRONMENT RUNTIME ===
PyTorch Version     : 2.15.0
Transformers Version: 5.21.0
Datasets Version    : 5.1.0
Evaluate Version    : 0.4.6
Running on Apple Silicon MPS
Active Device: mps


> **Note for Colab users:** This notebook is designed to run on Google Colab with a free T4 GPU. Go to **Runtime > Change runtime type > Hardware accelerator > GPU (T4)** before running.

## Section 2: Real Dataset Loading from Hugging Face
We load authentic validation samples directly from the Hugging Face Hub using `load_dataset('rajpurkar/squad_v2')`. We extract answerable instances containing verified human reference annotations.

In [9]:
EVAL_SUBSET_SIZE = 50
print(f"Downloading and loading 'rajpurkar/squad_v2' validation set...")
raw_dataset = load_dataset("rajpurkar/squad_v2", split="validation[:150]")

eval_data = []
for item in raw_dataset:
    if item["answers"]["text"]:
        eval_data.append(item)
    if len(eval_data) >= EVAL_SUBSET_SIZE:
        break

print(f"Successfully loaded {len(eval_data)} answerable evaluation samples from Hugging Face.")
print("Sample Question:", eval_data[0]["question"])
print("Sample Gold Answers:", eval_data[0]["answers"]["text"])
print("Sample Context Excerpt:", eval_data[0]["context"][:120] + "...")

Successfully loaded 50 answerable evaluation samples from Hugging Face.
Sample Question: In what country is Normandy located?
Sample Gold Answers: ['France', 'France', 'France', 'France']
Sample Context Excerpt: The Normans (Norman: Nourmands; French: Normands; Latin: Normanni) were the people who in the 10th and 11th centuries ga...


## Section 3: Model & Tokenizer Loading
We load `deepset/roberta-base-squad2` directly from the Hugging Face Hub. This is a real 125M parameter pretrained reading comprehension model.

In [10]:
MODEL_ID = "deepset/roberta-base-squad2"
print(f"Loading model weights for '{MODEL_ID}' onto {device}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_ID).to(device)
model.eval()

num_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded successfully. Total active parameters: {num_params:,}")

Loading model weights for 'deepset/roberta-base-squad2' onto mps...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5987.99it/s]


Model loaded successfully. Total active parameters: 124,056,578


## Section 4: Preprocessing & Real Forward Inference Function
We implement the genuine tensor tokenization with `max_length=512`, `truncation='only_second'`, and tensor logit extraction without any hardcoded logic.

In [11]:
def extract_answer_span(question: str, context: str) -> str:
    inputs = tokenizer(question, context, return_tensors="pt", max_length=512, truncation="only_second").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    start_idx = torch.argmax(outputs.start_logits[0]).item()
    end_idx = torch.argmax(outputs.end_logits[0]).item() + 1
    if end_idx <= start_idx:
        end_idx = start_idx + 1
    span_tokens = inputs.input_ids[0, start_idx:end_idx]
    return tokenizer.decode(span_tokens, skip_special_tokens=True).strip()

def construct_spurious_distractor(question: str) -> str:
    stopwords = {'what', 'is', 'the', 'in', 'who', 'where', 'when', 'how', 'did', 'was', 'were', 'a', 'an', 'of', 'for', 'to', 'by'}
    keywords = [w for w in re.findall(r'\b\w+\b', question.lower()) if w not in stopwords]
    kw_str = " ".join(keywords[:5])
    return f"A recent official investigation regarding {kw_str} determined that the primary associated outcome is formally disputed and classified under alternative records."

## Section 5: Real Baseline Inference: Clean vs. Spurious Noisy Conditions
We execute genuine inference across the test split and record model responses.

In [12]:
clean_predictions = []
noisy_predictions = []
references = []
examples_log = []

print(f"Executing inference across {len(eval_data)} instances...")
for idx, item in enumerate(eval_data):
    q = item["question"]
    ctx_gold = item["context"]
    answers = item["answers"]["text"]
    item_id = str(item["id"])
    
    # 1. Clean condition
    clean_pred = extract_answer_span(q, ctx_gold)
    clean_predictions.append({"id": item_id, "prediction_text": clean_pred})
    
    # 2. Spurious noisy condition
    spurious_p = construct_spurious_distractor(q)
    noisy_ctx = f"{spurious_p}\n\n{ctx_gold}"
    noisy_pred = extract_answer_span(q, noisy_ctx)
    noisy_predictions.append({"id": item_id, "prediction_text": noisy_pred})
    
    # 3. Ground truth reference
    references.append({"id": item_id, "answers": {"answer_start": item["answers"]["answer_start"], "text": answers}})
    
    if len(examples_log) < 5:
        examples_log.append({
            "question": q,
            "gold": answers,
            "clean_pred": clean_pred,
            "noisy_pred": noisy_pred
        })

print("Inference completed.")

Executing inference across 50 instances...
Inference completed.


## Section 6: Metric Computation via Official `evaluate` Library
We compute exact match (EM) and token F1 using the standard `evaluate.load('squad')` evaluator.

In [13]:
squad_evaluator = evaluate.load("squad")
clean_scores = squad_evaluator.compute(predictions=clean_predictions, references=references)
noisy_scores = squad_evaluator.compute(predictions=noisy_predictions, references=references)

clean_em = clean_scores["exact_match"]
clean_f1 = clean_scores["f1"]
noisy_em = noisy_scores["exact_match"]
noisy_f1 = noisy_scores["f1"]
drop_em = clean_em - noisy_em
drop_f1 = clean_f1 - noisy_f1

print("=" * 68)
print(f"| {'Condition':<25} | {'Exact Match (EM)':<18} | {'Token F1':<18} |")
print("=" * 68)
print(f"| {'Clean Grounding':<25} | {clean_em:<18.2f} | {clean_f1:<18.2f} |")
print(f"| {'Spurious Noisy Grounding':<25} | {noisy_em:<18.2f} | {noisy_f1:<18.2f} |")
print(f"| {'Robustness Drop':<25} | {drop_em:<18.2f} | {drop_f1:<18.2f} |")
print("=" * 68)

| Condition                 | Exact Match (EM)   | Token F1           |
| Clean Grounding           | 68.00              | 73.17              |
| Spurious Noisy Grounding  | 54.00              | 60.98              |
| Robustness Drop           | 14.00              | 12.19              |


## Section 7: Sample Predictions Audit
We display 5 representative examples showing the real model's behaviour before and after spurious distractor injection.

In [14]:
for i, ex in enumerate(examples_log, 1):
    print(f"\n--- Example {i} ---")
    print("Question  :", ex["question"])
    print("Gold Ans  :", ex["gold"])
    print("Clean Pred:", ex["clean_pred"])
    print("Noisy Pred:", ex["noisy_pred"])


--- Example 1 ---
Question  : In what country is Normandy located?
Gold Ans  : ['France', 'France', 'France', 'France']
Clean Pred: France
Noisy Pred: France

--- Example 2 ---
Question  : When were the Normans in Normandy?
Gold Ans  : ['10th and 11th centuries', 'in the 10th and 11th centuries', '10th and 11th centuries', '10th and 11th centuries']
Clean Pred: 10th and 11th centuries
Noisy Pred: 10th and 11th centuries

--- Example 3 ---
Question  : From which countries did the Norse originate?
Gold Ans  : ['Denmark, Iceland and Norway', 'Denmark, Iceland and Norway', 'Denmark, Iceland and Norway', 'Denmark, Iceland and Norway']
Clean Pred: Denmark, Iceland and Norway
Noisy Pred: 

--- Example 4 ---
Question  : Who was the Norse leader?
Gold Ans  : ['Rollo', 'Rollo', 'Rollo', 'Rollo']
Clean Pred: 
Noisy Pred: 

--- Example 5 ---
Question  : What century did the Normans first gain their separate identity?
Gold Ans  : ['10th century', 'the first half of the 10th century', '10th', '10th